# Pronóstico del consumo de agua de una planta con promedio móvil

## Introducción

Un **promedio móvil** calcula el promedio de las observaciones más recientes y lo desplaza a través del tiempo. Su principal utilidad es suavizar fluctuaciones aleatorias para observar mejor el nivel de una serie y estimar el corto plazo.

En este ejercicio analizaremos el consumo diario de agua de una planta industrial. El problema que buscamos resolver es: **¿cuántos metros cúbicos debemos considerar para planear el abastecimiento de los próximos días?**

Este caso es apropiado para un promedio móvil porque el consumo normal cambia lentamente, mientras que las lecturas diarias contienen ruido. Una ventana de 7 días permite estimar el nivel normal de consumo y evita sobrerreaccionar a una medición atípica.

## Objetivos del ejercicio

Al finalizar podremos:

1. Construir una serie diaria reproducible.
2. Identificar nivel, ruido y pequeñas variaciones de la demanda.
3. Calcular promedios móviles de 3, 7 y 14 días.
4. Usar un promedio móvil de 7 días para pronosticar.
5. Evaluar el resultado con métricas de error.
6. Interpretar cómo usar el resultado para planear abastecimiento y cuáles son sus limitaciones.

## 1. Preparar el entorno

Usaremos `pandas` para manipular la serie, `numpy` para generar datos reproducibles, `matplotlib` y `seaborn` para visualizar, y `scikit-learn` para calcular métricas. Todas son librerías populares disponibles en Google Colab.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.float_format', lambda value: f'{value:,.2f}')

## 2. Crear el dataset de consumo de agua

Para que el notebook sea autónomo, generaremos 180 días de consumo en metros cúbicos. La serie representa una planta con operación estable: combina un nivel base, una tendencia mínima y ruido moderado de medición. Este contexto es especialmente adecuado para un promedio móvil. La semilla fija permite que todos obtengan los mismos resultados al ejecutar el notebook.

In [ ]:
rng = np.random.default_rng(42)
fechas = pd.date_range('2025-01-01', periods=180, freq='D')
nivel_base = 500 + np.linspace(0, 2, len(fechas))
ruido = rng.normal(loc=0, scale=15, size=len(fechas))
consumo = np.maximum(100, nivel_base + ruido).round().astype(int)

datos = pd.DataFrame({'consumo_m3': consumo}, index=fechas)
datos.index.name = 'fecha'
datos.head()

## 3. Explorar el comportamiento del consumo

La gráfica muestra que las lecturas diarias fluctúan alrededor de un nivel relativamente estable. Esta es precisamente la situación donde un promedio móvil resulta útil: permite distinguir el consumo normal del ruido de una medición individual.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 8))
datos['consumo_m3'].plot(ax=axes[0], color='#2563eb', alpha=0.75)
axes[0].set_title('Consumo diario de agua de la planta')
axes[0].set_ylabel('Metros cúbicos')

datos['consumo_m3'].rolling(7).mean().plot(ax=axes[1], color='#16a34a', linewidth=2)
axes[1].axhline(datos['consumo_m3'].mean(), color='#f59e0b', linestyle='--', label='Promedio histórico')
axes[1].set_title('Nivel suavizado mediante promedio móvil de 7 días')
axes[1].set_ylabel('Metros cúbicos')
axes[1].set_xlabel('Fecha')
axes[1].legend()
plt.tight_layout()

## 4. Calcular promedios móviles

La función `rolling()` de `pandas` calcula ventanas móviles. Una ventana de 3 días responde más rápido, pero conserva más ruido; una ventana de 14 días suaviza más, aunque puede reaccionar tarde a cambios recientes. La ventana de 7 días es una elección natural porque representa una semana completa.

In [ ]:
datos['media_movil_3d'] = datos['consumo_m3'].rolling(window=3).mean()
datos['media_movil_7d'] = datos['consumo_m3'].rolling(window=7).mean()
datos['media_movil_14d'] = datos['consumo_m3'].rolling(window=14).mean()

plt.figure(figsize=(13, 6))
plt.plot(datos.index, datos['consumo_m3'], label='Consumo diario', alpha=0.35)
plt.plot(datos.index, datos['media_movil_3d'], label='Promedio móvil 3 días')
plt.plot(datos.index, datos['media_movil_7d'], label='Promedio móvil 7 días', linewidth=2)
plt.plot(datos.index, datos['media_movil_14d'], label='Promedio móvil 14 días', linewidth=2)
plt.title('Efecto de distintas ventanas de suavizamiento')
plt.ylabel('Metros cúbicos')
plt.xlabel('Fecha')
plt.legend()
plt.tight_layout()

## 5. Separar entrenamiento y prueba

Reservaremos los últimos 14 días para probar el pronóstico. La división respeta el orden temporal: el modelo solo podrá utilizar la información disponible antes del periodo que intentamos pronosticar.

In [ ]:
horizonte_prueba = 14
entrenamiento = datos['consumo_m3'].iloc[:-horizonte_prueba]
prueba = datos['consumo_m3'].iloc[-horizonte_prueba:]

print(f'Entrenamiento: {entrenamiento.index.min():%Y-%m-%d} a {entrenamiento.index.max():%Y-%m-%d}')
print(f'Prueba: {prueba.index.min():%Y-%m-%d} a {prueba.index.max():%Y-%m-%d}')

## 6. Construir el pronóstico con promedio móvil

Para pronosticar los próximos 14 días usaremos el promedio de los últimos 7 días disponibles. Después de cada día pronosticado, incorporaremos el valor pronosticado a la ventana; esto simula un pronóstico recursivo cuando todavía no conocemos los valores futuros.

In [ ]:
def pronostico_promedio_movil(historial, pasos, ventana=7):
    valores = list(historial.astype(float))
    pronosticos = []
    for _ in range(pasos):
        siguiente = np.mean(valores[-ventana:])
        pronosticos.append(siguiente)
        valores.append(siguiente)
    return pd.Series(pronosticos, index=prueba.index, name=f'Media móvil {ventana} días')

pronostico_mm = pronostico_promedio_movil(entrenamiento, horizonte_prueba, ventana=7)
pronostico_mm.head()

## 7. Comparar contra una línea base

La línea base repite el último valor observado. Esta comparación responde una pregunta clave: ¿el promedio móvil aporta información adicional frente a una regla todavía más simple?

In [ ]:
pronostico_base = pd.Series(entrenamiento.iloc[-1], index=prueba.index, name='Último valor')

plt.figure(figsize=(13, 6))
plt.plot(entrenamiento.index[-45:], entrenamiento.iloc[-45:], label='Histórico reciente', color='#64748b')
plt.plot(prueba.index, prueba, label='Real', color='#111827', linewidth=2)
plt.plot(prueba.index, pronostico_base, '--', label='Último valor', color='#ef4444')
plt.plot(prueba.index, pronostico_mm, '--', label='Promedio móvil 7 días', color='#16a34a', linewidth=2)
plt.axvline(prueba.index[0], color='black', linestyle=':', label='Inicio de prueba')
plt.title('Pronóstico de consumo: promedio móvil contra línea base')
plt.ylabel('Metros cúbicos')
plt.xlabel('Fecha')
plt.legend()
plt.tight_layout()

## 8. Evaluar los resultados

MAE indica cuántos metros cúbicos nos equivocamos en promedio; RMSE da más peso a errores grandes; MAPE expresa el error como porcentaje. En todas, un valor menor representa un pronóstico más cercano al consumo real.

In [ ]:
def calcular_metricas(real, pronostico):
    return pd.Series({
        'MAE': mean_absolute_error(real, pronostico),
        'RMSE': np.sqrt(mean_squared_error(real, pronostico)),
        'MAPE (%)': np.mean(np.abs((real - pronostico) / real)) * 100
    })

metricas = pd.DataFrame({
    'Último valor': calcular_metricas(prueba, pronostico_base),
    'Promedio móvil 7 días': calcular_metricas(prueba, pronostico_mm)
}).T
metricas

## Interpretación de resultados

La tabla permite determinar si el promedio móvil mejora la referencia. Si su MAE y MAPE son menores, el promedio de los últimos 7 días está representando mejor el nivel reciente que utilizar únicamente el último día. Si no la supera, puede significar que hubo un cambio brusco, una tendencia fuerte o que la ventana elegida no es adecuada.

También debemos observar que un promedio móvil suaviza por diseño: resulta útil para estimar el abastecimiento normal, pero puede reaccionar tarde ante una fuga, una parada de planta o un incremento extraordinario del consumo.

In [ ]:
mape_base = metricas.loc['Último valor', 'MAPE (%)']
mape_mm = metricas.loc['Promedio móvil 7 días', 'MAPE (%)']
mejora = (1 - mape_mm / mape_base) * 100

print(f'MAPE de la línea base: {mape_base:.2f}%')
print(f'MAPE del promedio móvil: {mape_mm:.2f}%')
if mejora > 0:
    print(f'El promedio móvil mejora el MAPE en {mejora:.2f}% frente a la línea base.')
else:
    print(f'El promedio móvil empeora el MAPE en {-mejora:.2f}% en este periodo de prueba.')
print('Decisión operativa: usar el promedio móvil como referencia de abastecimiento y revisar diariamente los desvíos.')

## Conclusiones

- El promedio móvil es útil cuando las lecturas diarias tienen ruido, pero el consumo normal cambia lentamente.
- Una ventana de 7 días permite estimar el abastecimiento normal de la planta sin depender de una sola lectura.
- Ventanas cortas reaccionan más rápido; ventanas largas producen una curva más estable, pero pueden retrasarse.
- El promedio móvil no conoce eventos futuros y puede fallar ante fugas, paros, mantenimientos o incrementos extraordinarios.
- Para operación, puede apoyar la planeación de agua y la detección de desvíos; para decisiones críticas conviene combinarlo con producción, turnos y eventos operativos.
- La ventana debe elegirse comparando varias alternativas mediante validación temporal, no solo por intuición.

## Pronóstico complementario: próximos 7 días

Hasta este punto evaluamos el promedio móvil sobre un periodo de prueba. Ahora utilizaremos toda la información disponible para estimar el consumo de los siguientes 7 días. Este resultado puede servir como referencia inmediata para planear el abastecimiento de agua.

In [ ]:
def pronostico_futuro_promedio_movil(historial, pasos, ventana=7):
    valores = list(historial.astype(float))
    pronosticos = []
    for _ in range(pasos):
        siguiente = np.mean(valores[-ventana:])
        pronosticos.append(siguiente)
        valores.append(siguiente)
    fechas_futuras = pd.date_range(
        start=historial.index.max() + pd.Timedelta(days=1),
        periods=pasos,
        freq='D'
    )
    return pd.Series(pronosticos, index=fechas_futuras, name='Pronóstico de consumo')

pronostico_7_dias = pronostico_futuro_promedio_movil(
    datos['consumo_m3'], pasos=7, ventana=7
)

pronostico_7_dias.round(2).to_frame()

### Visualización del pronóstico futuro

La siguiente gráfica conecta las últimas observaciones reales con los siete días pronosticados. La línea vertical marca el momento a partir del cual ya no tenemos valores observados y comienza la estimación.

In [ ]:
plt.figure(figsize=(13, 6))
historico_reciente = datos['consumo_m3'].tail(30)
plt.plot(historico_reciente.index, historico_reciente, label='Consumo real', color='#2563eb', marker='o')
plt.plot(pronostico_7_dias.index, pronostico_7_dias, label='Pronóstico próximos 7 días', color='#16a34a', marker='o', linewidth=2)
plt.axvline(datos.index.max(), color='black', linestyle=':', label='Último dato disponible')
plt.title('Consumo histórico reciente y pronóstico de los próximos 7 días')
plt.ylabel('Metros cúbicos')
plt.xlabel('Fecha')
plt.legend()
plt.tight_layout()

### Interpretación del pronóstico de 7 días

El promedio móvil genera una estimación estable porque cada nuevo día pronosticado utiliza el promedio de los últimos siete valores disponibles. Para abastecimiento, esto permite planear alrededor del consumo normal esperado.

La estimación no debe interpretarse como una predicción de eventos extraordinarios. Si se anuncia un mantenimiento, cambia el nivel de producción o se detecta una fuga, conviene complementar el pronóstico con información operativa y ajustar manualmente la decisión de abastecimiento.